# MCP(Tool 호출) 방식 시연 — PydanticAI Agent + @agent.tool_plain

MCP(Model Context Protocol) 방식은 LLM이 **사전에 정의된 Tool(함수)**을 자동으로 선택하고 호출하여 답변을 생성하는 구조입니다.
Agent가 질문을 분석하여 적절한 Tool을 골라 실행하고, 그 결과를 바탕으로 최종 답변을 구성합니다.

- **[장점]** 실시간 데이터 접근 / 외부 시스템 조작 / 환각 없음 (Tool 결과가 사실)
- **[단점]** Tool 수 증가 시 선택 정확도 저하 / 비정형 Q&A 부적합 / 사전 설계 필요

## 환경 설정

`.env` 파일에서 `OPENAI_API_KEY`를 로드하고, PydanticAI의 `Agent` 클래스를 임포트합니다.

In [1]:
from dotenv import load_dotenv
from pydantic_ai import Agent

# .env 파일에서 OPENAI_API_KEY 등 환경변수 로드
load_dotenv()

True

## Agent 정의

PydanticAI의 `Agent`를 생성합니다. `instructions`는 시스템 프롬프트 역할로,
Agent의 역할과 행동 지침을 정의합니다. 이후 등록할 Tool들은 이 Agent에 바인딩됩니다.

In [2]:
# PydanticAI Agent 생성 — instructions가 시스템 프롬프트 역할
agent = Agent(
    "openai:gpt-5.4",
    instructions=(
        "당신은 TechShop의 고객 상담 에이전트입니다. "
        "고객의 질문에 Tool을 활용하여 정확한 정보를 제공하세요."
    ),
)

## Tool 정의

`@agent.tool_plain` 데코레이터로 함수를 Tool로 등록합니다.
LLM은 **함수의 docstring을 읽고** 어떤 Tool을 호출할지 결정하므로, docstring이 정확할수록 선택 정확도가 올라갑니다.

- `tool_plain`: 의존성 주입 없이 단순 인자만 받는 Tool
- 함수명, 인자 타입, docstring 모두 LLM에게 전달됨
- 제품 데이터는 `data/products.json` 파일에서 로드 (hardcoded dict 대신 파일 기반 데이터소스)

In [3]:
import json
from pathlib import Path

# 제품 데이터를 JSON 파일에서 로드 (hardcoded dict 대신 파일 기반 데이터소스)
_DATA = json.loads(Path("data/products.json").read_text(encoding="utf-8"))


# Tool 1: 환불 정책 조회 — docstring이 LLM의 Tool 선택 기준이 된다
@agent.tool_plain
def get_refund_policy(product_id: str) -> str:
    """제품별 환불 정책을 조회한다. 환불 기간, 조건, 방법, 예외를 반환한다.
    고객이 환불/반품에 대해 질문할 때 사용한다."""
    policy = _DATA["refund_policies"].get(product_id)
    if policy:
        print(f"  [Tool] get_refund_policy('{product_id}') → 정책 조회 완료")
        return policy
    return f"제품 ID '{product_id}'에 대한 환불 정책을 찾을 수 없습니다."


# Tool 2: 주문 상태 조회 — 복합 질문 시 여러 Tool이 자동으로 호출됨
@agent.tool_plain
def get_order_status(order_id: str) -> str:
    """주문 번호로 주문 상태를 조회한다. 배송 현황, 예상 도착일을 반환한다.
    고객이 주문 상태나 배송에 대해 질문할 때 사용한다."""
    order = _DATA["orders"].get(order_id)
    if order:
        print(f"  [Tool] get_order_status('{order_id}') → 주문 조회 완료")
        return order
    return f"주문 번호 '{order_id}'를 찾을 수 없습니다."

## 시나리오 1: 환불 정책 질문 (단일 Tool 호출)

`await agent.run()` 한 줄로 전체 흐름이 자동 실행됩니다:

1. LLM이 질문을 분석하여 `get_refund_policy` Tool이 적합하다고 판단
2. `product_id="PROD-001"` 인자를 추출하여 Tool 호출
3. Tool 결과를 바탕으로 자연어 답변 생성

> **참고**: Jupyter 노트북은 이미 이벤트 루프가 실행 중이므로 `run_sync()` 대신 `await agent.run()`을 사용합니다.

In [4]:
# await agent.run() — Jupyter는 top-level await를 지원한다
# (run_sync()는 이미 실행 중인 이벤트 루프와 충돌하므로 사용 불가)
result = await agent.run("스마트 워치 Pro(PROD-001)의 환불 정책이 어떻게 되나요?")
print(f"[답변]\n{result.output}")

  [Tool] get_refund_policy('PROD-001') → 정책 조회 완료
[답변]
스마트 워치 Pro(PROD-001)의 환불 정책은 아래와 같습니다.

- 환불 기간: 구매 후 30일 이내
- 조건: 미개봉이며, 액세서리가 전부 포함되어 있어야 함
- 환불 방법: 원래 결제 수단으로 환불, 처리 기간은 3~5영업일
- 예외: 세일 기간에 구매한 경우 환불은 불가하고 교환만 가능합니다

원하시면 반품 절차도 간단히 안내해드릴게요.


## 시나리오 2: 주문 상태 + 환불 복합 질문 (다중 Tool 호출)

하나의 질문에 **여러 Tool이 필요한 경우**, LLM이 자동으로 두 Tool을 순서대로 호출합니다.
이 예시에서는 `get_order_status`로 배송 상태를 확인한 후, `get_refund_policy`로 환불 정책을 조회합니다.

In [5]:
# 복합 질문 — LLM이 get_order_status + get_refund_policy 두 Tool을 자동 호출
result = await agent.run(
    "주문번호 ORD-20260301 상태를 알려주세요. "
    "아직 안 왔으면 환불하고 싶은데, 환불 정책도 알려주세요."
)
print(f"[답변]\n{result.output}")

  [Tool] get_order_status('ORD-20260301') → 주문 조회 완료
[답변]
주문번호 `ORD-20260301` 확인 결과:

- 상품: 스마트 워치 Pro
- 상태: 배송 중
- 예상 도착일: 2026-03-05
- 송장번호: `KR1234567890`

현재는 아직 배송 완료 전이라 보입니다.

환불 정책은 상품별로 조회해야 하는데, 현재 주문 정보에서는 제품 ID가 확인되지 않아 정확한 정책 조회가 되지 않았습니다.  
스마트 워치 Pro의 **제품 ID**를 알려주시면 바로 환불 정책까지 정확히 안내해드릴게요.

참고로 보통은:
- 배송 전/상품 준비 단계: 취소 가능
- 배송 중/배송 완료 후: 반품(환불) 절차로 진행
- 상품별로 환불 가능 기간, 개봉 여부, 구성품 누락 여부 등에 따라 조건이 달라질 수 있습니다.

원하시면 제품 ID 알려주시면 바로 이어서 확인해드리겠습니다.
